In [12]:
import numpy as np
import cv2
import pynq
from pynq import Overlay
import urllib.request

print("=== STAP 1: Bitstream Laden ===")
overlay = Overlay("Edges.bit")
ip_core = overlay.vitis_convolution_0 

WIDTH = 128
HEIGHT = 128

# Lijst met 10 fixed URL's naar afbeeldingen
image_urls = [
    "https://picsum.photos/id/10/300/300",
    "https://picsum.photos/id/20/300/300",
    "https://picsum.photos/id/30/300/300",
    "https://picsum.photos/id/40/300/300",
    "https://picsum.photos/id/50/300/300",
    "https://picsum.photos/id/60/300/300",
    "https://picsum.photos/id/70/300/300",
    "https://picsum.photos/id/80/300/300",
    "https://picsum.photos/id/90/300/300",
    "https://picsum.photos/id/100/300/300"
]

print("=== STAP 2: Buffers Alloceren ===")
in_buffer = pynq.allocate(shape=(HEIGHT, WIDTH), dtype=np.uint8)
out_buffer = pynq.allocate(shape=(HEIGHT, WIDTH), dtype=np.int8)

# De twee verschillende kernels gedefinieerd
horizontal_kernel = [-1, -2, -1, 0, 0, 0, 1, 2, 1]
vertical_kernel   = [-1, 0, 1, -2, 0, 2, -1, 0, 1]

def run_hardware_convolution(kernel_matrix, output_filename):
    # Output buffer resetten voor een frisse run
    out_buffer[:] = 0
    out_buffer.flush()
    
    # Fysieke adressen en kernel registers koppelen
    ip_core.register_map.input_img = in_buffer.device_address
    ip_core.register_map.output_img = out_buffer.device_address
    
    ip_core.register_map.kernel_0_0 = kernel_matrix[0]
    ip_core.register_map.kernel_0_1 = kernel_matrix[1]
    ip_core.register_map.kernel_0_2 = kernel_matrix[2]
    ip_core.register_map.kernel_1_0 = kernel_matrix[3]
    ip_core.register_map.kernel_1_1 = kernel_matrix[4]
    ip_core.register_map.kernel_1_2 = kernel_matrix[5]
    ip_core.register_map.kernel_2_0 = kernel_matrix[6]
    ip_core.register_map.kernel_2_1 = kernel_matrix[7]
    ip_core.register_map.kernel_2_2 = kernel_matrix[8]
    
    # Start Hardware
    ip_core.register_map.CTRL.AP_START = 1
    while ip_core.register_map.CTRL.AP_IDLE == 0:
        pass
        
    out_buffer.invalidate()
    
    # Resultaat ophalen en indexverschuiving uit HLS corrigeren
    raw_data = out_buffer.copy()
    corrected_img = np.zeros((HEIGHT, WIDTH), dtype=np.uint8)
    corrected_img[1:HEIGHT-1, 1:WIDTH-1] = np.abs(raw_data[0:HEIGHT-2, 0:WIDTH-2])
    
    if np.max(corrected_img) > 0:
        corrected_img = cv2.normalize(corrected_img, None, 0, 255, cv2.NORM_MINMAX)
        
    cv2.imwrite(output_filename, corrected_img)
    print(f"    -> Opgeslagen: {output_filename}")


print("\n=== STAP 3: Start Duplo-Verwerking (Hor & Vert) van de 10 URL's ===")
for i, url in enumerate(image_urls):
    try:
        print(f"\n[Afbeelding {i+1}/10] Downloaden van: {url}")
        
        # Download en decodeer de afbeelding live in het geheugen
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            arr = np.asarray(bytearray(response.read()), dtype=np.uint8)
            img_raw = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
            
        if img_raw is None:
            print(f"  -> Fout: Kon afbeelding {i+1} niet decoderen.")
            continue

        # Resize naar de hardware-maat
        img_resized = cv2.resize(img_raw, (WIDTH, HEIGHT))
        
        # Duw de pixeldata in de input buffer
        in_buffer[:] = img_resized
        in_buffer.flush()
        
        # Run 1: Horizontale convolutie voor deze afbeelding
        run_hardware_convolution(horizontal_kernel, f"URL_{i+1}_Horizontal.png")
        
        # Run 2: Verticale convolutie voor deze afbeelding
        run_hardware_convolution(vertical_kernel, f"URL_{i+1}_Vertical.png")
        
    except Exception as e:
        print(f"  -> Fout bij verwerken van afbeelding {i+1}: {e}")

print("\n=== PROCES COMPLEET AFGEROND ===")
print("Alle 10 de afbeeldingen zijn verwerkt naar zowel een horizontale als verticale output!")

=== STAP 1: Bitstream Laden ===
=== STAP 2: Buffers Alloceren ===

=== STAP 3: Start Duplo-Verwerking (Hor & Vert) van de 10 URL's ===

[Afbeelding 1/10] Downloaden van: https://picsum.photos/id/10/300/300
    -> Opgeslagen: URL_1_Horizontal.png
    -> Opgeslagen: URL_1_Vertical.png

[Afbeelding 2/10] Downloaden van: https://picsum.photos/id/20/300/300
    -> Opgeslagen: URL_2_Horizontal.png
    -> Opgeslagen: URL_2_Vertical.png

[Afbeelding 3/10] Downloaden van: https://picsum.photos/id/30/300/300
    -> Opgeslagen: URL_3_Horizontal.png
    -> Opgeslagen: URL_3_Vertical.png

[Afbeelding 4/10] Downloaden van: https://picsum.photos/id/40/300/300
    -> Opgeslagen: URL_4_Horizontal.png
    -> Opgeslagen: URL_4_Vertical.png

[Afbeelding 5/10] Downloaden van: https://picsum.photos/id/50/300/300
    -> Opgeslagen: URL_5_Horizontal.png
    -> Opgeslagen: URL_5_Vertical.png

[Afbeelding 6/10] Downloaden van: https://picsum.photos/id/60/300/300
    -> Opgeslagen: URL_6_Horizontal.png
    -> Op